# ComfyUI Google Cloud TPU v3-8 (8-Core) Parallel Inference Benchmark

This notebook runs native Google Cloud TPU support in ComfyUI (`torch.device('tpu')` via `torch_tpu`), running 8 independent ComfyUI instances across the 8 TPU cores of a Google Cloud / Kaggle TPU v3-8 accelerator, executing parallel diffusion inference with Qwen-Image 2.1.

In [ ]:
import os
import sys
import traceback

# Ensure output directory exists immediately
OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOG_FILE = os.path.join(OUTPUT_DIR, "execution.log")
def log(msg):
    print(msg)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(str(msg) + "\n")

log(f"Python Version: {sys.version}")


In [ ]:
# Check accelerator and environment
import torch
log(f"PyTorch Version: {torch.__version__}")

# Install torch_tpu and gguf
!pip install -q torch_tpu --extra-index-url https://download.pytorch.org/whl/cpu >> /kaggle/working/output/pip_tpu.log 2>&1
!pip install -q gguf >> /kaggle/working/output/pip_tpu.log 2>&1

try:
    import torch_tpu
    log("torch_tpu successfully imported!")
except Exception as e:
    log(f"Notice on torch_tpu import: {e}")

has_acc = hasattr(torch, 'accelerator') and torch.accelerator.is_available()
log(f"torch.accelerator available: {has_acc}")
if has_acc:
    log(f"Current accelerator: {torch.accelerator.current_accelerator()}")
    log(f"Accelerator device count: {torch.accelerator.device_count()}")


In [ ]:
# Clone ComfyUI feature branch into /tmp/ComfyUI (keeping /kaggle/working clean!)
COMFY_DIR = "/tmp/ComfyUI"
if os.path.exists(COMFY_DIR):
    !rm -rf {COMFY_DIR}

!git clone --depth 1 -b feat/tpu-multi-core-support https://github.com/vuthanhtrung2010/ComfyUI.git {COMFY_DIR}
log(f"Cloned ComfyUI to {COMFY_DIR}")

# Install ComfyUI dependencies into environment
!pip install -q -r {COMFY_DIR}/requirements.txt >> /kaggle/working/output/pip_reqs.log 2>&1

# Install ComfyUI-GGUF custom node for Qwen-Image 2.1 GGUF
!git clone https://github.com/city96/ComfyUI-GGUF.git {COMFY_DIR}/custom_nodes/ComfyUI-GGUF
!sed -i 's/"qwen_image"/"qwen_image", "qwen_image21"/g' {COMFY_DIR}/custom_nodes/ComfyUI-GGUF/loader.py
!sed -i 's/raise ValueError(f"This model is not currently supported - ({e})")/arch_str = "qwen_image"/g' {COMFY_DIR}/custom_nodes/ComfyUI-GGUF/loader.py
log("ComfyUI dependencies and ComfyUI-GGUF custom node installed.")


In [ ]:
# Download Qwen-Image 2.1 model checkpoints into /tmp/ComfyUI/models/
!mkdir -p {COMFY_DIR}/models/unet {COMFY_DIR}/models/clip {COMFY_DIR}/models/vae

log("Downloading Qwen-Image 2.1 models (UNet GGUF, CLIP, VAE)...")

# 1. Diffusion Model (Qwen-Image 2.1 Turbo Quantized GGUF ~5.5GB)
!wget -c -q --show-progress -O {COMFY_DIR}/models/unet/qwen-image-2.1-UC-Q4_K_M.gguf \
  "https://huggingface.co/abenzerps/Qwen-Image-2.1-Turbo-Quantized/resolve/main/qwen-image-2.1-turbo-Q4_K_M.gguf"

# 2. Text Encoder CLIP (~8.8GB)
!wget -c -q --show-progress -O {COMFY_DIR}/models/clip/qwen3vl_8b_int8_convrot.safetensors \
  "https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-Quantized/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors"

# 3. VAE (~645MB)
!wget -c -q --show-progress -O {COMFY_DIR}/models/vae/qwen_image_2.1_vae_bf16.safetensors \
  "https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-Quantized/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors"

!ls -lh {COMFY_DIR}/models/unet/ {COMFY_DIR}/models/clip/ {COMFY_DIR}/models/vae/
log("Models downloaded successfully.")


In [ ]:
# Launch 8 ComfyUI instances across the 8 TPU cores
sys.path.insert(0, COMFY_DIR)
from tools.tpu_orchestrator import TPUClusterManager

log_cluster_dir = os.path.join(COMFY_DIR, "logs/tpu_cluster")
manager = TPUClusterManager(cores=8, base_port=8188, comfy_root=COMFY_DIR)
manager.start(log_dir=log_cluster_dir)

log("Waiting for all 8 TPU instances to become ready...")
ready = manager.wait_ready(timeout=180)
log(f"Cluster ready state: {ready}")


In [ ]:
# Execute parallel 8-core benchmark
import json
import time

workflow_path = os.path.join(COMFY_DIR, "workflows/qwen_image_tpu_test.json")
with open(workflow_path, "r", encoding="utf-8") as f:
    wf = json.load(f)
prompt_graph = wf.get("prompt", wf)

subjects = [
    "nebula galaxy exploding with celestial turquoise dust",
    "cyberpunk cityscape at night with glowing neon reflections",
    "ancient mystical temple hidden in lush mountain rainforest",
    "futuristic solar sailing ship skimming atmospheric plasma",
    "bioluminescent coral reef teeming with radiant aquatic life",
    "crystal ice fortress beneath a shimmering aurora borealis",
    "enchanted giant redwood forest illuminated by glowing spirits",
    "sleek retro-futuristic landspeeder drifting across desert dunes"
]

prompts = []
for i, subj in enumerate(subjects):
    p = json.loads(json.dumps(prompt_graph))
    p["4"]["inputs"]["text"] = f"{subj}, hyper-detailed digital art masterpiece 8k"
    p["7"]["inputs"]["seed"] = 1000 + i
    prompts.append(p)

log("Dispatching 8 parallel image generation jobs across all 8 TPU cores...")
t0 = time.time()
results = manager.dispatch_parallel(prompts, timeout=600)
elapsed = time.time() - t0

log(f"Parallel Batch Completed in {elapsed:.2f} seconds!")
log(f"Successful Prompts: {results['successful_prompts']} / {results['total_prompts']}")
log(f"Throughput: {results['throughput_images_per_min']} images/minute")


In [ ]:
# Save benchmark report and output images directly to /kaggle/working/output
import shutil

with open(os.path.join(OUTPUT_DIR, "benchmark_report.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

comfy_out = os.path.join(COMFY_DIR, "output")
if os.path.exists(comfy_out):
    for fname in os.listdir(comfy_out):
        if fname.endswith((".png", ".jpg", ".webp")):
            shutil.copy(os.path.join(comfy_out, fname), os.path.join(OUTPUT_DIR, fname))

log("Saved final artifacts to /kaggle/working/output:")
!ls -lh /kaggle/working/output/

manager.stop()
log("All 8 TPU ComfyUI instances cleanly terminated.")
